# Online Distillation v2: Optimized for Blackwell

**Optimizations**:
- Pre-generate teacher corpus with `model.generate()` + KV cache (10x faster)
- fp16 mixed precision training
- Top-K distillation (KL on top-50 logits, not 151K)
- Batch=64 (Blackwell has the VRAM)
- Separate data-gen and training phases

**Expected**: ~30-40 min total on Blackwell

In [ ]:
!pip install -q torch transformers datasets safetensors sentencepiece

In [ ]:
import math, os, time, json, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
from dataclasses import dataclass

device = "cuda"
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

## 1. Student model (80M)

In [ ]:
CFG = {
    "vocab_size": 151936, "hidden_size": 384,
    "num_attention_heads": 6, "num_key_value_heads": 2,
    "intermediate_size": 1536, "num_hidden_layers": 6,
    "rms_norm_eps": 1e-6, "rope_theta": 1000000.0,
    "max_seq_len": 128, "tie_word_embeddings": True,
}

@dataclass
class C:
    vocab_size:int=151936; hidden_size:int=384; num_attention_heads:int=6
    num_key_value_heads:int=2; intermediate_size:int=1536; num_hidden_layers:int=6
    rms_norm_eps:float=1e-6; rope_theta:float=1e6; max_seq_len:int=128
    tie_word_embeddings:bool=True

def rope(hd,sl,th):
    f=1./(th**(torch.arange(0,hd,2).float()/hd))
    e=torch.cat([torch.outer(torch.arange(sl).float(),f)]*2,-1)
    return e.cos(),e.sin()

def rot(x):
    d=x.shape[-1]//2; return torch.cat([-x[...,d:],x[...,:d]],-1)

class RN(nn.Module):
    def __init__(s,d,e=1e-6): super().__init__(); s.w=nn.Parameter(torch.ones(d)); s.e=e
    def forward(s,x): return s.w*(x*torch.rsqrt(x.pow(2).mean(-1,keepdim=True)+s.e))

class At(nn.Module):
    def __init__(s,c):
        super().__init__()
        s.nh,s.nk=c.num_attention_heads,c.num_key_value_heads
        s.hd=c.hidden_size//c.num_attention_heads; s.ng=s.nh//s.nk
        s.q=nn.Linear(c.hidden_size,s.nh*s.hd,bias=True)
        s.k=nn.Linear(c.hidden_size,s.nk*s.hd,bias=True)
        s.v=nn.Linear(c.hidden_size,s.nk*s.hd,bias=True)
        s.o=nn.Linear(s.nh*s.hd,c.hidden_size,bias=False)
    def forward(s,x,co,si,m):
        B,L,_=x.shape
        q=s.q(x).view(B,L,s.nh,s.hd).transpose(1,2)
        k=s.k(x).view(B,L,s.nk,s.hd).transpose(1,2)
        v=s.v(x).view(B,L,s.nk,s.hd).transpose(1,2)
        q=q*co+rot(q)*si; k=k*co+rot(k)*si
        if s.ng>1:
            k=k.unsqueeze(2).expand(B,s.nk,s.ng,L,s.hd).reshape(B,s.nh,L,s.hd)
            v=v.unsqueeze(2).expand(B,s.nk,s.ng,L,s.hd).reshape(B,s.nh,L,s.hd)
        a=F.softmax(torch.matmul(q,k.transpose(-2,-1))/math.sqrt(s.hd)+m,-1)
        return s.o(torch.matmul(a,v).transpose(1,2).contiguous().view(B,L,-1))

class FF(nn.Module):
    def __init__(s,c):
        super().__init__()
        s.g=nn.Linear(c.hidden_size,c.intermediate_size,bias=False)
        s.u=nn.Linear(c.hidden_size,c.intermediate_size,bias=False)
        s.d=nn.Linear(c.intermediate_size,c.hidden_size,bias=False)
    def forward(s,x): return s.d(F.silu(s.g(x))*s.u(x))

class Bk(nn.Module):
    def __init__(s,c):
        super().__init__()
        s.at=At(c);s.ff=FF(c);s.n1=RN(c.hidden_size,c.rms_norm_eps);s.n2=RN(c.hidden_size,c.rms_norm_eps)
    def forward(s,x,co,si,m):
        x=x+s.at(s.n1(x),co,si,m); return x+s.ff(s.n2(x))

class Draft(nn.Module):
    def __init__(s,c):
        super().__init__()
        s.cfg=c; s.emb=nn.Embedding(c.vocab_size,c.hidden_size)
        s.layers=nn.ModuleList([Bk(c) for _ in range(c.num_hidden_layers)])
        s.norm=RN(c.hidden_size,c.rms_norm_eps)
        s.head=nn.Linear(c.hidden_size,c.vocab_size,bias=False)
        if c.tie_word_embeddings: s.head.weight=s.emb.weight
        hd=c.hidden_size//c.num_attention_heads
        co,si=rope(hd,c.max_seq_len,c.rope_theta)
        s.register_buffer('rc',co.unsqueeze(0).unsqueeze(0))
        s.register_buffer('rs',si.unsqueeze(0).unsqueeze(0))
        mk=torch.triu(torch.full((c.max_seq_len,c.max_seq_len),-1e9),diagonal=1)
        s.register_buffer('mk',mk.unsqueeze(0).unsqueeze(0))
    def forward(s,ids):
        h=s.emb(ids)
        for l in s.layers: h=l(h,s.rc,s.rs,s.mk)
        return s.head(s.norm(h))

student = Draft(C(**CFG)).to(device)
print(f"Student: {sum(p.numel() for p in student.parameters()):,} params")

## 2. Teacher + tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

teacher = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", torch_dtype=torch.float16).to(device).eval()
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f"Teacher: {sum(p.numel() for p in teacher.parameters()):,} params (fp16)")

## 3. Phase 1: Pre-generate teacher corpus

Generate 50K sequences of 128 tokens each. Store tokens + top-50 logit indices/values.

In [ ]:
from datasets import load_dataset

ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
seeds = []
for text in ds["text"]:
    text = text.strip()
    if len(text) > 50:
        ids = tokenizer.encode(text, add_special_tokens=False)
        if len(ids) >= 10:
            seed_len = random.randint(8, min(25, len(ids)))
            seeds.append(ids[:seed_len])
    if len(seeds) >= 100000:
        break
print(f"Seed prompts: {len(seeds):,}")

In [ ]:
SEQ = 128
TOP_K = 50  # only store top-50 logits per position
N_SEQS = 50_000
GEN_BATCH = 128  # big batch for fast generation

all_token_ids = []   # (N_SEQS, SEQ)
all_top_indices = [] # (N_SEQS, SEQ, TOP_K)
all_top_values = []  # (N_SEQS, SEQ, TOP_K)

t0 = time.time()
n_batches = (N_SEQS + GEN_BATCH - 1) // GEN_BATCH

print(f"Generating {N_SEQS:,} sequences with teacher (batch={GEN_BATCH})...")

for batch_i in range(n_batches):
    bs = min(GEN_BATCH, N_SEQS - len(all_token_ids))
    if bs <= 0: break

    # Pick random seeds, pad to same length
    batch_seeds = random.sample(seeds, bs)
    max_seed = max(len(s) for s in batch_seeds)
    # Pad seeds
    padded_seeds = []
    attn_masks = []
    for s in batch_seeds:
        pad_len = max_seed - len(s)
        # Left-pad with pad_token for generation
        padded_seeds.append([tokenizer.pad_token_id] * pad_len + s)
        attn_masks.append([0] * pad_len + [1] * len(s))

    input_ids = torch.tensor(padded_seeds, device=device)
    attn_mask = torch.tensor(attn_masks, device=device)

    # Generate with teacher using HF generate (fast, uses KV cache)
    gen_len = SEQ - max_seed
    with torch.no_grad():
        out = teacher.generate(
            input_ids, attention_mask=attn_mask,
            max_new_tokens=max(1, gen_len),
            do_sample=False,  # greedy
            pad_token_id=tokenizer.pad_token_id,
        )

    # Truncate/pad to exactly SEQ tokens
    seqs = out[:, -SEQ:]  # take last SEQ tokens
    if seqs.shape[1] < SEQ:
        pad = torch.full((bs, SEQ - seqs.shape[1]), tokenizer.pad_token_id, device=device)
        seqs = torch.cat([pad, seqs], dim=1)

    # Get teacher logits for the full sequence
    with torch.no_grad(), autocast(device_type="cuda", dtype=torch.float16):
        t_logits = teacher(seqs).logits.float()  # (bs, SEQ, V)

    # Extract top-K indices and values (HUGE memory savings: 50 vs 151936)
    top_vals, top_idx = torch.topk(t_logits, TOP_K, dim=-1)  # (bs, SEQ, TOP_K)

    all_token_ids.append(seqs.cpu())
    all_top_indices.append(top_idx.cpu().to(torch.int32))
    all_top_values.append(top_vals.cpu().to(torch.float16))

    if (batch_i + 1) % 20 == 0:
        n = len(all_token_ids) * GEN_BATCH
        el = time.time() - t0
        eta = el / (batch_i + 1) * (n_batches - batch_i - 1)
        print(f"  {n:,}/{N_SEQS:,} seqs | {el/60:.1f}m | ETA {eta/60:.0f}m")

# Concatenate
all_token_ids = torch.cat(all_token_ids, dim=0)[:N_SEQS]
all_top_indices = torch.cat(all_top_indices, dim=0)[:N_SEQS]
all_top_values = torch.cat(all_top_values, dim=0)[:N_SEQS]

print(f"\nGenerated {all_token_ids.shape[0]:,} sequences in {(time.time()-t0)/60:.1f} min")
print(f"  token_ids:   {all_token_ids.shape}")
print(f"  top_indices: {all_top_indices.shape} ({all_top_indices.nbytes/1e9:.1f} GB)")
print(f"  top_values:  {all_top_values.shape} ({all_top_values.nbytes/1e9:.1f} GB)")

## 4. Phase 2: Train with top-K distillation + mixed precision

In [ ]:
# Free teacher from GPU memory — we only need the pre-generated data now
del teacher
torch.cuda.empty_cache()
print(f"Freed teacher. GPU mem: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
STEPS = 50_000
BATCH = 64
LR = 5e-4
TEMP = 1.5
ALPHA = 0.8

# DataLoader from pre-generated data
dataset = TensorDataset(all_token_ids, all_top_indices, all_top_values)
loader = DataLoader(dataset, batch_size=BATCH, shuffle=True, drop_last=True, num_workers=2, pin_memory=True)

opt = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)
scaler = GradScaler()
WU = 1000

def lr_fn(s):
    if s < WU: return s / WU
    return 0.5 * (1 + math.cos(math.pi * (s - WU) / (STEPS - WU)))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)


def topk_kl_loss(student_logits, top_indices, top_values, temperature):
    """
    KL divergence only on the teacher's top-K tokens.
    Much cheaper than full-vocab KL (50 vs 151936).
    """
    B, L, V = student_logits.shape
    K = top_indices.shape[-1]

    # Gather student logits at teacher's top-K positions
    # top_indices: (B, L, K) — need to index into (B, L, V)
    s_topk = torch.gather(student_logits, -1, top_indices.long())  # (B, L, K)

    # Teacher probabilities from top-K values
    t_probs = F.softmax(top_values.float() / temperature, dim=-1)  # (B, L, K)

    # Student log-probs at those positions
    s_log_probs = F.log_softmax(s_topk / temperature, dim=-1)  # (B, L, K)

    # KL divergence
    kl = F.kl_div(s_log_probs, t_probs, reduction="batchmean") * temperature**2
    return kl


student.train()
it = iter(loader)
losses = []
best = float('inf')
t0 = time.time()

print(f"Training: {STEPS:,} steps | batch={BATCH} | lr={LR} | fp16 | top-{TOP_K} KL")
print(f"Data: {all_token_ids.shape[0]:,} seqs x {SEQ} tokens")
print("-" * 70)

for step in range(1, STEPS + 1):
    try:
        tok_ids, t_idx, t_val = next(it)
    except StopIteration:
        it = iter(loader)
        tok_ids, t_idx, t_val = next(it)

    tok_ids = tok_ids.to(device)
    t_idx = t_idx.to(device)
    t_val = t_val.to(device)

    inp = tok_ids[:, :-1]
    labels = tok_ids[:, 1:]
    t_idx_shifted = t_idx[:, :-1]  # teacher logits predict next token
    t_val_shifted = t_val[:, :-1]

    B, L = inp.shape
    if L < SEQ:
        s_inp = F.pad(inp, (0, SEQ - L))
    else:
        s_inp = inp[:, :SEQ]; labels = labels[:, :SEQ]
        t_idx_shifted = t_idx_shifted[:, :SEQ]; t_val_shifted = t_val_shifted[:, :SEQ]
        L = SEQ

    with autocast(device_type="cuda", dtype=torch.float16):
        s_logits = student(s_inp)[:, :L, :]

        # Top-K KL loss
        kl = topk_kl_loss(s_logits, t_idx_shifted, t_val_shifted, TEMP)

        # CE loss (hard targets)
        V = s_logits.size(-1)
        ce = F.cross_entropy(s_logits.reshape(-1, V), labels.reshape(-1))

        loss = ALPHA * kl + (1 - ALPHA) * ce

    opt.zero_grad()
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()
    sched.step()

    losses.append(loss.item())

    if step % 500 == 0:
        avg = sum(losses[-500:]) / 500
        el = time.time() - t0
        eta = el / step * (STEPS - step)
        tag = " *" if avg < best else ""
        if avg < best: best = avg
        tps = step * BATCH * L / el
        print(f"Step {step:6d}/{STEPS:,} | loss={avg:.4f}{tag} | lr={sched.get_last_lr()[0]:.1e} | {tps/1e6:.1f}M tok/s | {el/60:.0f}m | ETA {eta/60:.0f}m")

    if step % 10_000 == 0:
        from safetensors.torch import save_file as sf
        os.makedirs("ckpt", exist_ok=True)
        st = {k: v.cpu() for k, v in student.state_dict().items() if not k.startswith(("rc","rs","mk"))}
        sf(st, f"ckpt/step_{step}.safetensors")
        print(f"  >> ckpt saved")

print(f"\nDone! {(time.time()-t0)/60:.0f} min | best={best:.4f}")

## 5. Eval: acceptance rate

In [ ]:
# Reload teacher for eval
teacher = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", torch_dtype=torch.float32).to(device).eval()
student.eval()

prompts = [
    "The meaning of life is",
    "Once upon a time",
    "The capital of France is",
    "In machine learning, a neural network",
    "The president of the United States",
    "Water boils at a temperature of",
    "The largest planet in the solar system",
    "Python is a programming language that",
    "Albert Einstein was born in",
    "The quick brown fox jumps over",
]

total_m, total_t = 0, 0
for prompt in prompts:
    ids = tokenizer.encode(prompt)
    sc, tc = list(ids), list(ids)
    for _ in range(50):
        # Student
        p = sc[-SEQ:] + [0]*max(0,SEQ-len(sc))
        with torch.no_grad(): sl = student(torch.tensor([p[:SEQ]],device=device))
        sc.append(sl[0,min(len(sc),SEQ)-1].argmax().item())
        # Teacher
        with torch.no_grad(): tl = teacher(torch.tensor([tc],device=device)).logits
        tc.append(tl[0,-1].argmax().item())

    match = sum(1 for a,b in zip(sc[len(ids):],tc[len(ids):]) if a==b)
    total_m += match; total_t += 50
    print(f'  [{match:2d}/50={match/50:3.0%}] "{prompt}"')
    print(f'    S: {tokenizer.decode(sc[len(ids):])[:60]}')
    print(f'    T: {tokenizer.decode(tc[len(ids):])[:60]}')

print(f"\n{'='*60}")
print(f"  OVERALL ACCEPTANCE: {total_m}/{total_t} = {total_m/total_t:.0%}")
print(f"  Target: >30% for useful speculative decoding")
print(f"{'='*60}")

## 6. Save

In [ ]:
from safetensors.torch import save_file
os.makedirs("draft_80m_v2", exist_ok=True)
state = {k: v.cpu() for k, v in student.state_dict().items() if not k.startswith(("rc","rs","mk"))}
save_file(state, "draft_80m_v2/draft_80m.safetensors")
with open("draft_80m_v2/config.json", "w") as f:
    json.dump(CFG, f, indent=2)
print(f"Saved: {os.path.getsize('draft_80m_v2/draft_80m.safetensors')/1e6:.0f} MB")

In [ ]:
try:
    from google.colab import files
    files.download("draft_80m_v2/draft_80m.safetensors")
    files.download("draft_80m_v2/config.json")
except:
    print("Files in draft_80m_v2/")